In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("../Airlinedataset.csv")

# Features used by the validated pricing model
features = [
    "days_to_departure",
    "seats_remaining",
    "historical_demand",
    "competitor_price",
    "booking_velocity",
    "is_weekend",
    "flight_capacity"
]

target = "ticket_price"

X = df[features]
y = df[target]

# Same train/test split used throughout the project
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Train validated pricing model
model = LinearRegression()
model.fit(X_train, y_train)

print("Pricing model loaded successfully.")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Pricing model loaded successfully.
Training rows: 800
Test rows: 200


In [2]:
base_flight = {
    "days_to_departure": 30,
    "seats_remaining": 90,
    "historical_demand": 60,
    "competitor_price": 6250,
    "booking_velocity": 16,
    "is_weekend": 0,
    "flight_capacity": 180
}

base_data = pd.DataFrame([base_flight])

dynamic_price = model.predict(base_data[features])[0]

print(f"Model-recommended price: ₹{dynamic_price:,.2f}")

Model-recommended price: ₹9,214.99


In [3]:
def estimate_expected_bookings(
    base_bookings,
    price,
    reference_price,
    elasticity
):
    """
    Estimate bookings under a simulated price-response assumption.

    This is a scenario model, not an empirically estimated demand model.
    """

    price_ratio = price / reference_price

    expected_bookings = base_bookings * (price_ratio ** elasticity)

    return max(expected_bookings, 0)

In [4]:
# Simulation assumptions
base_bookings = 90
reference_price = 9214.99

# Negative elasticity:
# higher prices -> lower expected bookings
elasticity = -1.2

expected_bookings = estimate_expected_bookings(
    base_bookings=base_bookings,
    price=dynamic_price,
    reference_price=reference_price,
    elasticity=elasticity
)

expected_revenue = dynamic_price * expected_bookings

print(f"Simulated expected bookings: {expected_bookings:.2f}")
print(f"Simulated expected revenue: ₹{expected_revenue:,.2f}")

Simulated expected bookings: 90.00
Simulated expected revenue: ₹829,349.11


In [5]:
# Static pricing benchmark
static_price = reference_price

# Simulate bookings and revenue for static pricing
static_bookings = estimate_expected_bookings(
    base_bookings=base_bookings,
    price=static_price,
    reference_price=reference_price,
    elasticity=elasticity
)

static_revenue = static_price * static_bookings

# Simulate bookings and revenue for dynamic pricing
dynamic_bookings = estimate_expected_bookings(
    base_bookings=base_bookings,
    price=dynamic_price,
    reference_price=reference_price,
    elasticity=elasticity
)

dynamic_revenue = dynamic_price * dynamic_bookings

comparison = pd.DataFrame([
    {
        "strategy": "Static Pricing",
        "price": static_price,
        "expected_bookings": static_bookings,
        "expected_revenue": static_revenue
    },
    {
        "strategy": "Dynamic Pricing",
        "price": dynamic_price,
        "expected_bookings": dynamic_bookings,
        "expected_revenue": dynamic_revenue
    }
])

comparison

,strategy,price,expected_bookings,expected_revenue
0,Static Pricing,9214.990000,90.000000,829349.100000
1,Dynamic Pricing,9214.989307,90.000008,829349.112467


In [6]:
comparison["expected_bookings"] = comparison["expected_bookings"].round(2)
comparison["expected_revenue"] = comparison["expected_revenue"].round(2)

comparison

,strategy,price,expected_bookings,expected_revenue
0,Static Pricing,9214.990000,90.0,829349.10
1,Dynamic Pricing,9214.989307,90.0,829349.11


In [7]:
scenarios = pd.DataFrame([
    {
        "scenario": "Low Demand",
        "days_to_departure": 45,
        "seats_remaining": 150,
        "historical_demand": 30,
        "competitor_price": 5500,
        "booking_velocity": 5,
        "is_weekend": 0,
        "flight_capacity": 180
    },
    {
        "scenario": "Normal Demand",
        "days_to_departure": 30,
        "seats_remaining": 90,
        "historical_demand": 60,
        "competitor_price": 6250,
        "booking_velocity": 16,
        "is_weekend": 0,
        "flight_capacity": 180
    },
    {
        "scenario": "High Demand",
        "days_to_departure": 5,
        "seats_remaining": 20,
        "historical_demand": 90,
        "competitor_price": 7000,
        "booking_velocity": 28,
        "is_weekend": 1,
        "flight_capacity": 180
    }
])

# Predict dynamic prices
scenarios["dynamic_price"] = model.predict(
    scenarios[features]
)

# Use the same static benchmark for every scenario
scenarios["static_price"] = reference_price

# Simulate bookings under each strategy
scenarios["static_bookings"] = scenarios["seats_remaining"]

scenarios["dynamic_bookings"] = scenarios.apply(
    lambda row: estimate_expected_bookings(
        base_bookings=row["seats_remaining"],
        price=row["dynamic_price"],
        reference_price=reference_price,
        elasticity=elasticity
    ),
    axis=1
)

# Prevent simulated bookings from exceeding available seats
scenarios["dynamic_bookings"] = np.minimum(
    scenarios["dynamic_bookings"],
    scenarios["seats_remaining"]
)

# Calculate revenue
scenarios["static_revenue"] = (
    scenarios["static_price"] *
    scenarios["static_bookings"]
)

scenarios["dynamic_revenue"] = (
    scenarios["dynamic_price"] *
    scenarios["dynamic_bookings"]
)

scenarios[
    [
        "scenario",
        "static_price",
        "dynamic_price",
        "static_bookings",
        "dynamic_bookings",
        "static_revenue",
        "dynamic_revenue"
    ]
]

,scenario,static_price,dynamic_price,static_bookings,dynamic_bookings,static_revenue,dynamic_revenue
0,Low Demand,9214.99,6574.361330,150,150.000000,1382248.5,986154.199503
1,Normal Demand,9214.99,9214.989307,90,90.000000,829349.1,829349.037663
2,High Demand,9214.99,12905.738746,20,13.350096,184299.8,172292.853463


In [8]:
results = scenarios[
    [
        "scenario",
        "static_price",
        "dynamic_price",
        "static_bookings",
        "dynamic_bookings",
        "static_revenue",
        "dynamic_revenue"
    ]
].copy()

results["revenue_difference"] = (
    results["dynamic_revenue"] -
    results["static_revenue"]
)

results["revenue_change_pct"] = (
    results["revenue_difference"] /
    results["static_revenue"] * 100
)

results.round(2)

,scenario,static_price,dynamic_price,static_bookings,dynamic_bookings,static_revenue,dynamic_revenue,revenue_difference,revenue_change_pct
0,Low Demand,9214.99,6574.36,150,150.00,1382248.5,986154.20,-396094.30,-28.66
1,Normal Demand,9214.99,9214.99,90,90.00,829349.1,829349.04,-0.06,-0.00
2,High Demand,9214.99,12905.74,20,13.35,184299.8,172292.85,-12006.95,-6.51


In [9]:
elasticities = [-0.5, -1.0, -1.2, -1.5, -2.0]

sensitivity_results = []

for elasticity_value in elasticities:

    for _, row in scenarios.iterrows():

        dynamic_bookings = estimate_expected_bookings(
            base_bookings=row["seats_remaining"],
            price=row["dynamic_price"],
            reference_price=reference_price,
            elasticity=elasticity_value
        )

        dynamic_bookings = min(
            dynamic_bookings,
            row["seats_remaining"]
        )

        dynamic_revenue = (
            row["dynamic_price"] *
            dynamic_bookings
        )

        sensitivity_results.append({
            "elasticity": elasticity_value,
            "scenario": row["scenario"],
            "dynamic_price": row["dynamic_price"],
            "expected_bookings": dynamic_bookings,
            "expected_revenue": dynamic_revenue
        })

elasticity_results = pd.DataFrame(sensitivity_results)

elasticity_results.round(2)

,elasticity,scenario,dynamic_price,expected_bookings,expected_revenue
0,-0.5,Low Demand,6574.36,150.00,986154.20
1,-0.5,Normal Demand,9214.99,90.00,829349.04
2,-0.5,High Demand,12905.74,16.90,218106.63
3,-1.0,Low Demand,6574.36,150.00,986154.20
4,-1.0,Normal Demand,9214.99,90.00,829349.04
5,-1.0,High Demand,12905.74,14.28,184299.80
6,-1.2,Low Demand,6574.36,150.00,986154.20
7,-1.2,Normal Demand,9214.99,90.00,829349.04
8,-1.2,High Demand,12905.74,13.35,172292.85
9,-1.5,Low Demand,6574.36,150.00,986154.20


In [10]:
elasticity_summary = (
    elasticity_results
    .pivot(
        index="scenario",
        columns="elasticity",
        values="expected_revenue"
    )
    .round(2)
)

elasticity_summary

elasticity,-2.0,-1.5,-1.2,-1.0,-0.5
scenario,,,,,
High Demand,131594.23,155733.08,172292.85,184299.80,218106.63
Low Demand,986154.20,986154.20,986154.20,986154.20,986154.20
Normal Demand,829349.04,829349.04,829349.04,829349.04,829349.04


## Revenue Simulation Findings

A scenario-based revenue simulation was developed to compare static and model-based dynamic pricing.

Because the dataset does not contain observed booking responses to different ticket prices, price elasticity was not estimated directly from the data. Instead, the simulation uses explicitly stated hypothetical elasticity values.

The simulation was evaluated across multiple elasticity assumptions:

- -0.5
- -1.0
- -1.2
- -1.5
- -2.0

This sensitivity analysis demonstrates how strongly simulated revenue depends on the assumed relationship between ticket price and expected bookings.

The results should therefore be interpreted as scenario analysis rather than measured revenue improvement.

The dynamic pricing model provides the predicted ticket price based on the observed flight-state variables. The subsequent booking and revenue calculations are simulated using the assumed demand-response function.

### Important Limitation

The available dataset does not contain:

- historical booking counts at different observed prices,
- rejected/accepted booking attempts,
- timestamps for sequential booking events,
- or experimental price variation.

Therefore, the project cannot identify a causal price elasticity or claim that dynamic pricing would produce a specific real-world revenue increase.

The revenue simulation is used to demonstrate how a pricing recommendation could be evaluated under different demand-response assumptions.